In [ ]:
#Import necessary modules
import pandas as pd
import numpy as np
import ast
from scipy.stats import ranksums,wilcoxon
#kmeans
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import statistics
from collections import Counter

import math
from scipy.stats import variation
from scipy.stats import iqr
from scipy import stats 

import random
from random import seed
from random import randint
from sklearn.neighbors import LocalOutlierFactor
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from sklearn.metrics import confusion_matrix
import scipy.stats
import scipy.stats as st
from scipy.stats import t

import pandas as pd
from sklearn.ensemble import IsolationForest
from statistics import median
# from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
patients=[1135, 1450, 1464, 1497, 1504, 
          1511, 1541, 1559, 1572, 1582, 
          1586, 1603, 1607, 1609, 1615, 
          1617, 1628, 1657, 1660, 1706]

In [ ]:
# exit_rooms = ['Hallway','Front Door','Back Door','Other Door',
#               'Balcony Door','Other Door 2','Garage Door','Car 1','Balcony 1']#'Bathroom'

In [ ]:
exit_rooms = ['Front Door','Back Door','Other Door',
              'Balcony Door','Other Door 2','Garage Door','Balcony 1']#'Bathroom'

In [ ]:
def readData(patient):
    tm = pd.read_csv(f"{patient}_cleaned_removedays"'.csv') #solved double coverage problem
    out = np.array(tm).tolist()  

    for i in range(len(out)):
        if 'Bedroom' in out[i][1]:
            out[i][1]='Bedroom'
            
        if 'Bathroom' in out[i][1]:
            out[i][1]='Bathroom'
    
        if 'Sensor Line' in out[i][1] or 'Extra sensor line' in out[i][1]:
            out[i][1]='Hallway'
            
        if 'Hallway' in out[i][1]:
            out[i][1]= 'Hallway'
            
        if 'Walk-in Closet' in out[i][1]:
            out[i][1]= 'Walk-in Closet'
            
        if 'Kitchen' in out[i][1]:
            out[i][1]= 'Kitchen'
            
        if 'Dining Room' in out[i][1]:
            out[i][1]= 'Dining Room'
            
#         if 'Lounge' in out[i][1]:
#             out[i][1]= 'Lounge'
    
    
    return out

In [ ]:
def possible_visitor_non_adjacents(patient):
    try:
        tm = pd.read_csv(f"{patient}_possible_visitor_non_adjacent_twice_alldata"'.csv')
        possible_visitor_non_adjacent = np.array(tm).tolist() 
    except:
        possible_visitor_non_adjacent =[]
    return possible_visitor_non_adjacent

In [ ]:
# def possible_visitor_adjacents(patient):
#     try:
#         tm = pd.read_csv(f"{patient}_possible_visitor_visitor_adjacent_twice_alldata"'.csv')
#         possible_visitor_adjacent = np.array(tm).tolist()  
#     except:
#         possible_visitor_adjacent =[]
#     return possible_visitor_adjacent

### 4. Count sensor firings

In [ ]:
# find relavant hour of the day
def possible_visitor_hour(activity):                 
    from datetime import datetime        
#             print(out[exit_all[i][1]:exit_all[j][1]],exit_all[i][1],exit_all[j][1])
    record_time=[]
    for item in out:
        t0=item[0].split(' ')
        record_time.append(t0[0])
    record_time=list(set(record_time))
    record_time=sorted(record_time, key=lambda x: datetime.strptime(x, "%d/%m/%Y").strftime("%m/%d/%Y"))
#     record_time


    new_info={}

    for each in record_time:
        new_info[each]=[]
        for item in activity:
            time=item[0][0]
            to=time.split(' ')
            if to[0]==each:
                new_info[each].append(item)

    #Calculate active time 
    tm=[]
    import datetime
    for value in new_info.values():
        for item in value:
            t0=item[0][0].split(' ')
            t1=item[1][0].split(' ')
            start=t0[1]
            end =t1[1]
            start_dt = datetime.datetime.strptime(start, '%H:%M:%S')
            end_dt = datetime.datetime.strptime(end, '%H:%M:%S')
            
            #find satrt date and end date
            start_dt_d = datetime.datetime.strptime(t0[0],"%d/%m/%Y")
            end_dt_d = datetime.datetime.strptime(t1[0],"%d/%m/%Y")
#             print('end_dt',end_dt_d,'start_dt',start_dt_d,(end_dt_d-start_dt_d).days)

            #(1) activity period within an hour
            if start_dt_d==end_dt_d and start_dt.hour == end_dt.hour:
                #  calculate inactive time
                diff=(end_dt-start_dt).seconds
                h = (str(start_dt.hour))
                tm.append([t0[0],[t0[1],t1[1]],h,diff])

            #(2) if within same day active over an hour boundary
            if end_dt.hour!=start_dt.hour and start_dt_d==end_dt_d:
                if end_dt.hour !=0 :
                    diff = (end_dt-start_dt).seconds
                    h = (str(start_dt.hour))
                    tm.append([t0[0],[t0[1],t1[1]],h,diff])

                    for i in range(1,end_dt.hour-start_dt.hour):
                        hm = start_dt.hour+i
                        h2 = (str(hm))
                        tm.append([t0[0],[t0[1],t1[1]],h2,diff])

                    h2 = (str(end_dt.hour))
                    tm.append([t0[0],[t0[1],t1[1]],h2,diff])


            #(3) acrocss days  

            if start_dt_d!=end_dt_d:
                # calculate time difference
                diff = (end_dt-start_dt).seconds
               
               #  <1> start time               
                h1 = (str(start_dt.hour))
                # start time
                tm.append([t0[0],[t0[1],t1[1]],h1,diff])
                
                
                # other possible hours on start day
                for i in range(start_dt.hour+1,24):
                    hm = i
                    h2 = (str(hm))

                    tm.append([t0[0],[t0[1],t1[1]],h2,diff])
                
                
                
                
                #  <2>other possible in between date
                if  (end_dt_d-start_dt_d).days>1: # more than 24 hrs
                    overlap_day = (end_dt_d-start_dt_d).days
                    for d in range(1, overlap_day):
                        dt = datetime.datetime.strptime(t0[0], "%d/%m/%Y")
                        add_d = datetime.timedelta(days=d)
                        new_dt=(dt+add_d).strftime("%d/%m/%Y")             
#                         print(new_dt)
                        
                        for i in range(0,24):
                            hm = i
                            h2 = (str(hm))
                            tm.append([new_dt, [t0[1],t1[1]],h2,diff])
                
                
                
                 #<3>  other possible hours on end day
                for i in range(0, end_dt.hour):
                    hm = i
                    h2 = (str(hm))
#                         print(h2)
                    tm.append([t1[0],[t0[1],t1[1]],h2,diff])

    
    
               #<3.1> end time
                h2 = (str(end_dt.hour))
                tm.append([t1[0], [t0[1],t1[1]],h2,diff])

    # store info 
    possible_days=[]

    for each in tm:
        # check if greater than 1 minutes
        
#         if each[3]>=1*60:     
#             print(each)
        possible_days.append(each)
    from datetime import datetime
    possible_days.sort(key=lambda x: (datetime.strptime(x[0], "%d/%m/%Y").strftime("%m/%d/%Y")))
 
    return possible_days  


In [ ]:
def visitor_days(possible_days):
    new=[]
    new2=[]
    for each in possible_days:
        new.append([each[0],each[2]])
        
    for each in new:
        if each not in new2:
            new2.append(each)
    return new2



In [ ]:
def firing_data_hrs(out):
    # Create an empty dictionary to store the firing data
    firing_data = {}

    # Loop through each data point and extract the date and hour of the firing
    for point in out:
       
        datetime = point[0]

        # Extract the date and hour from the datetime string
        date, time = datetime.split()
        hour = time.split(":")[0]

        # If this is the first firing for this date, create a new dictionary for it
        if date not in firing_data:
            firing_data[date] = [0 for h in range(24)]
            
        if point[1] in exit_rooms:
            firing_data[date][int(hour)] = 1
    return firing_data


### split data into train and test

In [ ]:
# find date in data

def date_data(out):
    # number of sensors in each house
    all_date=[]
    for i in range(len(out)):
        dt = out[i][0].split(' ')[0]
        all_date.append(dt)

    #remove duplicate
    new_date=list(dict.fromkeys(all_date))
    return new_date

In [ ]:
# split data into train and test---60days
def split_tain_test(n_init):  
    import datetime
    traning_data = []
    test_data = []
    for item in out:
        start_date =datetime.datetime.strptime(out[0][0].split(' ')[0], "%d/%m/%Y").date()   
        dt =datetime.datetime.strptime(item[0].split(' ')[0], "%d/%m/%Y").date()
        
        #last traning date
        last=new_date[n_init]
        
        last_dt =datetime.datetime.strptime(last, "%d/%m/%Y").date()
        if dt < last_dt:
#             print(start_date,dt)
            traning_data.append(item)
        else:
            test_data.append(item)
            
    return traning_data, test_data


### significant firings

In [ ]:
def count_transitions(data):
    import datetime
    hours=[i for i in range(0,24)]

    # 创建一个空字典，用于存储每天每小时的数据
    hour_counts = {}

    previous_room = None

    # 遍历每个数据点
    for d in data:
        
        # 将时间戳转换为日期和小时
        date = datetime.datetime.strptime(d[0].split(' ')[0], "%d/%m/%Y").date()
        hour = datetime.datetime.strptime(d[0].split(' ')[1], "%H:%M:%S").hour

        # 检查字典中是否已存在该日期和小时的数据
        if date not in hour_counts:
            hour_counts[date] = {}
            
        for h in hours:
            if h not in hour_counts[date]:
                if hour ==h:  
                    hour_counts[date][hour] = 0
                else:
                    hour_counts[date][h] = 0
        
        room = d[1]

        if previous_room is None:
            previous_room = room

            continue
        if room != previous_room :
#             print(previous_room,room,state)
            hour_counts[date][hour] +=1
            
        previous_room = room

        # 将数据点添加到字典中
    #     hourly_data[date][hour].append(value)
    return hour_counts

In [ ]:
def count_firings(data):
    import datetime
    hours=[i for i in range(0,24)]

    # 创建一个空字典，用于存储每天每小时的数据
    hour_counts = {}

    # 遍历每个数据点
    for d in data:
        # 将时间戳转换为日期和小时
        date = datetime.datetime.strptime(d[0].split(' ')[0], "%d/%m/%Y").date()
        hour = datetime.datetime.strptime(d[0].split(' ')[1], "%H:%M:%S").hour

        # 检查字典中是否已存在该日期和小时的数据
        if date not in hour_counts:
            hour_counts[date] = {}

        for h in hours:
            if h not in hour_counts[date]:
                if hour ==h:  
                    hour_counts[date][hour] = 0
                else:
                    hour_counts[date][h] = 0
        
        hour_counts[date][hour] += 1
        # 将数据点添加到字典中
    #     hourly_data[date][hour].append(value)
    return hour_counts

In [ ]:
def remove_left_home(hour_counts):# 遍历 left 列表中的项
    from datetime import datetime
    # Create a set of (date, hour) tuples from the list
    list_keys = {(datetime.strptime(date, '%d/%m/%Y').date(), int(hour)) for date, _, hour, _, _ in left}

    # Remove dictionary entries with the same (date, hour)
    for date in list(hour_counts.keys()):
        for hour in list(hour_counts[date].keys()):
            if (date, hour) in list_keys:
                if hour_counts[date][hour]==0: #如果hourly_data对应的hour数值大于0，则不删除
                   # The total absence from the house is in the complete hours when there was nothing.
                    del hour_counts[date][hour]
            
    return hour_counts


In [ ]:
def hourly_firing(hour_counts):
    firings_hour={}
    hours=[i for i in range(0,24)]

    for a in hours:
        firings_hour[a]=[]

    for a in hours:
        for time, count in hour_counts.items():
            for hr,firings in count.items():
                if hr==a:
                    firings_hour[a].append(firings)
    return firings_hour            

In [ ]:
def significant_dt(hour_counts,outliers_thresholds):
    significant_dt_hrs=[]
    from datetime import datetime
    for dt, count in hour_counts.items():
        for hr,firings in count.items():
    #         print(dt,hr,firings)
            for hours, sig_firings in outliers_thresholds.items():
    #             print(hours, firings)
                if hr==hours:
                    for each in sig_firings:
                        if each ==firings:
                            date_obj = datetime.strptime(str(dt), '%Y-%m-%d')
                            formatted_date = date_obj.strftime('%d/%m/%Y')
    #                         print(dt,hr,firings)
                            significant_dt_hrs.append([formatted_date,hr])
    return significant_dt_hrs



In [ ]:
def possible(possible_visitor_non_adjacent,num):
    data_dict = {}
    # loop over each day

    for day in new_date:  
        if day not in data_dict:
            data_dict[day] = {h: 0 for h in range(24)}
            for date, hour in possible_visitor_non_adjacent:
                if date  in data_dict:
                
                    data_dict[date][int(hour)] = num

    return data_dict

In [ ]:
def read_left(patient):
    threshold=1800
    try:
        #file.csv is an empty csv file
        tm = pd.read_csv(f"{patient,threshold}_left_days_outing_living"'.csv')
        # Convert 'Date' to datetime format
        tm['Date'] = pd.to_datetime(tm['Date'])
        tm['Date'] =tm['Date'].dt.strftime('%d/%m/%Y')
        left_days_hours = np.array(tm).tolist() 
    
    except pd.errors.EmptyDataError:
        left_days_hours =[]
    return left_days_hours

In [ ]:
# split data into train and test---60days
def split_tain_test(n_init):  
    import datetime
    traning_data = []
    test_data = []
    for item in out:
        start_date =datetime.datetime.strptime(out[0][0].split(' ')[0], "%d/%m/%Y").date()   
        dt =datetime.datetime.strptime(item[0].split(' ')[0], "%d/%m/%Y").date()
        
        #last traning date
        last=new_date[n_init]
        
        last_dt =datetime.datetime.strptime(last, "%d/%m/%Y").date()
        if dt < last_dt:
#             print(start_date,dt)
            traning_data.append(item)
        else:
            test_data.append(item)
            
    return traning_data, test_data


In [ ]:
def threshold_Isolation_forest(percentile,train_data,data):
    outliers_IF_testing = {}
    hours=[i for i in range(0,24)]

    for a in hours:
        #convert data type for training
        data_all= np.array(sorted(data[a])).astype(int).reshape(-1, 1)

        # Isolation forest--each hour
        model = IsolationForest(n_estimators=100, max_samples="auto",random_state=66)
        model.fit(data_all)


        # <2> original way to make a prediction   
#         
        

# #         #2.1<Hu's paper>
#         # Calculate mean and standard deviation of isolation scores
#         isolation_scores= abs(model.score_samples(data_all)) #isolation_scores:isolation score of each instance
#         print(isolation_scores)
        
#         mis = np.mean(isolation_scores)
#         stis = np.std(isolation_scores)

#         # Calculate the threshold as MIS + 2 * STIS
#         threshold = mis + 2 * stis
#         outliers = data_all[isolation_scores > threshold]
#         print('hours',a,len(data_all),isolation_scores,'threshold',threshold,'outliers',outliers)

       
         #2.2 <based by the paper>  #outliers in testing 输出异常值
        isolation_scores= abs(model.score_samples(data_all))
        y_pred_testing =  model.predict(data_all)
        outliers = data_all[y_pred_testing == -1]
        outliers = [int(x) for x in outliers.flatten()]
         #remove lower than median/70 percentile---cannot be significant increase
        training=np.array(sorted(train_data[a])).astype(int).reshape(-1, 1)
        threshold=round(np.percentile(training, percentile))
        
        outliers=[x for x in outliers if x>threshold]
        print('hours',a,len(data_all),data_all,'threshold',threshold,'outliers',outliers)
        
        outliers_IF_testing[a]=outliers
        
    return outliers_IF_testing

In [ ]:

percentiles=[50,75,90,95,99]#,75,90,95,99
for num in patients:
    patient = num
    out=readData(patient)
    possible_visitor_non_adjacent=possible_visitor_non_adjacents(patient)
#     possible_visitor_adjacent=possible_visitor_adjacents(patient)

    #find when user left home
    left=read_left(patient)
    new_date=date_data(out)
    
    #entrance firings
    firing_data=firing_data_hrs(out)
    #save entrance_firing to csv  
    df_firing=pd.DataFrame.from_dict(firing_data)
    df_firing.to_csv(f"{patient}_entrance_firing_alldata"'.csv', index=False, header=True)
    
    # train_weeks = 10

    n_init=60

    split_output=split_tain_test(n_init)
    # train_data = split_tain_test[0]
    train_data = split_output[0]
    # test = split_tain_test[1]
    test_data = split_output[1]



    train_date=date_data(train_data)
    test_date=date_data(test_data)
    new_date=date_data(out)


    #(transition)
    hour_counts_transitions=remove_left_home(count_transitions(out))
    hour_transitions=hourly_firing(hour_counts_transitions)
    hourly_transitions = np.array(hour_transitions).tolist()  
    #training-transition
    hour_counts_train_transitions=remove_left_home(count_transitions(train_data))
    firings_hour_train_transitions=hourly_firing(hour_counts_train_transitions)
    hourly_train_transitions = np.array(firings_hour_train_transitions).tolist()  
   
    
    
     #(firings)
    hour_counts_firings=remove_left_home(count_firings(out))
    firings_hour=hourly_firing(hour_counts_firings)
    hourly_firings = np.array(firings_hour).tolist()  
    #training-firings
    hour_counts_train=remove_left_home(count_firings(train_data))
    firings_hour_train=hourly_firing(hour_counts_train)
    hourly_firings_train = np.array(firings_hour_train).tolist()  
 
    
    for per in percentiles:
       #threshold (use training)
        outliers_IF_transitions=threshold_Isolation_forest(per,hourly_train_transitions,hourly_transitions)
        significant_dts_transitions=significant_dt(hour_counts_transitions,outliers_IF_transitions)

        #threshold (use training) and outliers for testing
        outliers_IF_firings=threshold_Isolation_forest(per,hourly_firings_train,hourly_firings)
        significant_dts_firings=significant_dt(hour_counts_firings,outliers_IF_firings)



        
        #define different number for visualisation
        data_dict_non_adjacent=possible(possible_visitor_non_adjacent,0.95)
#         data_dict_adjacent=possible(possible_visitor_adjacent,0.98)
        
        data_dict_significant_dts_firings=possible(significant_dts_firings,1)
        data_dict_significant_dts_transitions=possible(significant_dts_transitions,0.9)

#         #save outputs
        a=data_dict_non_adjacent
        b=data_dict_significant_dts_firings
        c=data_dict_significant_dts_transitions

    

        from collections import defaultdict

        # 将所有的字典的hour和date提取出来
        all_hours = defaultdict(list)
        for d in [b, c]:
            for date, hours in d.items():
                for hour, value in hours.items():
                    if value != 0:
                        all_hours[date].append(hour)

        # 统计满足条件的hour of the day
        result = defaultdict(dict)
        for date, hours in all_hours.items():
            for hour in hours:
                if hours.count(hour) > 0:
                    result[date][hour] = hours.count(hour)
                else:
                    result[date][hour] = 0


        #save three_consitions to csv
        df =  pd.DataFrame.from_dict(result, orient='index').T
        # # sort columns by key of values
        df = df.apply(lambda x: pd.Series(x.dropna().to_dict())).sort_index()

        df.to_csv(f"{patient,per}_possible_visitors_two_conditions_IF_alldata"'.csv', index=False, header=True)



    

    #save ground truth (non_adjacent) to csv
    data_dict_non_adjacent=possible(possible_visitor_non_adjacent,1)
    df_ground=pd.DataFrame.from_dict(data_dict_non_adjacent, orient='index').T
    df_ground.to_csv(f"{patient}_non_adjacent_twice_alldata"'.csv', index=False, header=True)

# #     #save ground truth to csv
#     data_dict_adjacent=possible(possible_visitor_adjacent,1)
#     df_ground_adjacent=pd.DataFrame.from_dict(data_dict_adjacent, orient='index').T
#     df_ground_adjacent.to_csv(f"{patient}_adjacent_twice_alldata"'.csv', index=False, header=True)


In [ ]:
x